# PDCP · N06 — E2 counterfactual experiment (+ mean-edge-length edits)
**Notebook 6 of 13.** Kaggle: **GPU T4 x2** · Internet **On** · **Save & Run All (Commit)**.
Expected ≈ 15 quota-hours in total → **two sessions**: the first stops cleanly at ~11.3 h; save its output as a dataset (e.g. `kbs-n6-output`), set `RESUME_ROOT` to it and run again (≈ 3–4 h).

**Inputs:** N03 output (pairs, drawings, images) and N05 output (E1 results, used for the base answers in the summary). Optional `RESUME_ROOT`.

**Why mean edge length was added (from the E1 cue analysis, 18 Nov 2026)**
- In the main set, the lower-stress drawing has the shorter mean edge length in **91 %** of pairs (Spearman ρ = −0.59 with Δstress): an un-planned fourth proxy.
- A multivariate choice model shows Qwen3-VL-4B picks the drawing with **longer** edges (z = +8.7), which explains its below-chance accuracy; InternVL and Qwen2.5-VL prefer smaller/compact drawings.
- E2 therefore edits **four** cues one at a time on drawing A, with stress and the other cues fixed:
  NU +0.05/+0.10 · ELD +0.05/+0.10/+0.15 · XR +0.02/+0.04/+0.06 (from N03) and **MEL −0.05/+0.05/+0.10** (generated here, utils v0.1.3; feasibility 99 % in testing).
- Every counterfactual pair is queried in **both orders**, so the effect is a change in order-averaged P(choose A) — position bias cancels.

**Outputs** (`/kaggle/working/n06_output/`): `pdcp_utils.py` v0.1.3, `mel_cf_drawings.parquet`, `mel_cf_pairs.parquet`, `images/768/CFM_*.png`, `e2_queries.csv`, `results/e2/<model>.jsonl`, `e2_effects.csv`, `e2_status.json`, figures, logs, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N05_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n5-output"
RESUME_ROOT = None        # e.g. "/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-output" for the second session
OUTPUT_DIR = "/kaggle/working/n06_output"
MODELS = ["Q25VL-7B", "Q3VL-4B", "Q3VL-2B", "InternVL-4B", "Gemma3-4B"]   # Q25VL alone ~13 GPU-h -> gets its own GPU
MEL_DELTAS = [-0.05, 0.05, 0.10]
PROMPT = "P-Def"
MAX_NEW_TOKENS = 96
SESSION_HOURS = 11.3
SEED = 2026
N_CPU = 4

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
T0 = time.time()
for d in ["results/e2", "logs", "images/768"]: os.makedirs(f"{OUTPUT_DIR}/{d}", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
N_GPU = torch.cuda.device_count(); print("GPUs:", N_GPU); assert N_GPU >= 1, "Set Accelerator = GPU T4 x2"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.57", "accelerate", "bitsandbytes",
                "qwen-vl-utils", "num2words", "joblib"], capture_output=True, text=True)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN found")
except Exception as e:
    print("No HF_TOKEN -> Gemma skipped:", type(e).__name__)
if HF_TOKEN is None: MODELS = [m for m in MODELS if m != "Gemma3-4B"]
print("models:", MODELS)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

GPUs: 2
No HF_TOKEN -> Gemma skipped: BackendError
models: ['Q25VL-7B', 'Q3VL-4B', 'Q3VL-2B', 'InternVL-4B']


## 1. Shared module v0.1.3 (adds mean edge length `mel` as metric + optimisation target)

In [3]:
%%writefile /kaggle/working/n06_output/pdcp_utils.py
"""
pdcp_utils.py  —  shared code for the PDCP project (Proxy-Decoupled Counterfactual Probing)
Version: 0.1.0  (written by N01, 2026-11)

Contents
  1. Graph families            make_graph(family, n, seed)
  2. Exact metrics             ksm, node_uniformity, edge_length_deviation, crossing_ratio, all_metrics
  3. Differentiable surrogates (torch)  used only inside the optimiser
  4. PDSG Stage 1              pdsg_optimise(G, targets, ...)
  5. Rendering                 render_png(G, X, path, ...)
  6. Parsing model output      parse_answer(text)
  7. Statistics helpers        bootstrap_ci(...)

All coordinates X are numpy arrays of shape (n, 2). Node order = sorted(G.nodes()) after relabel to 0..n-1.
"""
from __future__ import annotations

import json
import math
import re
import time
from dataclasses import dataclass, field

import networkx as nx
import numpy as np

__version__ = "0.1.3"

# ----------------------------------------------------------------------------
# 1. Graph families
# ----------------------------------------------------------------------------
FAMILY_SIZES = {
    "ER": [10, 25, 50, 100],
    "BA": [25, 50, 100],
    "SBM": [25, 50, 100],
    "WS": [25, 50],
    "GRID": [16, 25, 49, 100],
    "TREE": [25, 50, 100],
}


def _connected_or_retry(fn, seed, max_tries=200):
    for t in range(max_tries):
        G = fn(seed + 1000 * t)
        if nx.is_connected(G):
            return G
    raise RuntimeError("could not sample a connected graph")


def make_graph(family: str, n: int, seed: int) -> nx.Graph:
    """Return a connected simple graph with nodes relabelled 0..n-1."""
    family = str(family).upper(); n = int(n); seed = int(seed)
    if family == "ER":
        p = 3.5 / (n - 1)  # expected average degree 3.5
        G = _connected_or_retry(lambda s: nx.gnp_random_graph(n, p, seed=s), seed)
    elif family == "BA":
        G = nx.barabasi_albert_graph(n, 2, seed=seed)
    elif family == "SBM":
        k = {25: 2, 50: 3, 100: 4}.get(n, max(2, n // 25))
        p_in = {25: 0.40, 50: 0.25, 100: 0.15}.get(n, 0.2)
        sizes = [n // k] * k
        sizes[-1] += n - sum(sizes)
        P = [[p_in if i == j else 0.02 for j in range(k)] for i in range(k)]
        G = _connected_or_retry(lambda s: nx.stochastic_block_model(sizes, P, seed=s), seed)
        G = nx.Graph(G)
    elif family == "WS":
        G = nx.connected_watts_strogatz_graph(n, 4, 0.1, seed=seed)
    elif family == "GRID":
        a = int(round(math.sqrt(n)))
        G = nx.grid_2d_graph(a, a)
    elif family == "TREE":
        try:
            G = nx.random_labeled_tree(n, seed=seed)
        except AttributeError:  # older networkx
            G = nx.random_tree(n, seed=seed)
    else:
        raise ValueError(f"unknown family {family}")
    G = nx.convert_node_labels_to_integers(G, ordering="sorted")
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


def graph_arrays(G: nx.Graph):
    """Edge index array (m,2), shortest-path matrix D (n,n), and index arrays for i<j pairs."""
    n = G.number_of_nodes()
    E = np.array(sorted((min(u, v), max(u, v)) for u, v in G.edges()), dtype=np.int64)
    D = np.zeros((n, n), dtype=np.float64)
    for s, lengths in nx.all_pairs_shortest_path_length(G):
        for t, d in lengths.items():
            D[s, t] = d
    iu, ju = np.triu_indices(n, k=1)
    return E, D, iu, ju


def nonadjacent_edge_pairs(E: np.ndarray):
    """All pairs of edges that share no endpoint (the pairs that can cross)."""
    m = len(E)
    a, b = np.triu_indices(m, k=1)
    e1, e2 = E[a], E[b]
    share = (e1[:, 0] == e2[:, 0]) | (e1[:, 0] == e2[:, 1]) | (e1[:, 1] == e2[:, 0]) | (e1[:, 1] == e2[:, 1])
    return a[~share], b[~share]


# ----------------------------------------------------------------------------
# 2. Exact metrics
# ----------------------------------------------------------------------------
def normalise(X: np.ndarray) -> np.ndarray:
    """Uniform scale into the unit box, aspect ratio preserved, centred."""
    X = np.asarray(X, dtype=np.float64)
    X = X - X.min(0)
    s = X.max()
    if s <= 0:
        return np.full_like(X, 0.5)
    X = X / s
    X = X + (1.0 - X.max(0)) / 2.0
    return X


def _pav(y: np.ndarray, w: np.ndarray | None = None) -> np.ndarray:
    """Pool-adjacent-violators: isotonic (non-decreasing) fit of y in given order."""
    y = np.asarray(y, dtype=np.float64)
    w = np.ones_like(y) if w is None else np.asarray(w, dtype=np.float64)
    vals, wts, cnt = [], [], []
    for yi, wi in zip(y, w):
        vals.append(yi); wts.append(wi); cnt.append(1)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            v2, w2, c2 = vals.pop(), wts.pop(), cnt.pop()
            v1, w1, c1 = vals.pop(), wts.pop(), cnt.pop()
            wt = w1 + w2
            vals.append((v1 * w1 + v2 * w2) / wt); wts.append(wt); cnt.append(c1 + c2)
    return np.repeat(vals, cnt)


def ksm(X: np.ndarray, D: np.ndarray, iu=None, ju=None, ties: str = "secondary") -> float:
    """
    Kruskal (non-metric) stress in [0,1]; dhat = isotonic regression of e on d.
    ties='secondary' (default): all pairs with equal graph distance get the same dhat
        -> identical to sklearn.isotonic.IsotonicRegression, the convention used by Mooney et al.
    ties='primary': Kruskal's primary approach (ties may be broken); gives lower values. Kept for reference only.
    """
    n = len(X)
    if iu is None:
        iu, ju = np.triu_indices(n, k=1)
    d = D[iu, ju]
    e = np.linalg.norm(X[iu] - X[ju], axis=1)
    if ties == "primary":
        order = np.lexsort((e, d))
        dhat = np.empty_like(e)
        dhat[order] = _pav(e[order])
    else:
        levels, inv, cnt = np.unique(d, return_inverse=True, return_counts=True)
        means = np.bincount(inv, weights=e) / cnt
        fit = _pav(means, cnt)
        dhat = fit[inv]
    num = np.sum((e - dhat) ** 2)
    den = np.sum(e ** 2)
    return float(np.sqrt(num / den)) if den > 0 else 0.0


def node_uniformity(X: np.ndarray) -> float:
    """0 = nodes spread uniformly over a ceil(sqrt n)^2 grid on the drawing's bounding box, 1 = all in one cell."""
    n = len(X)
    g = int(math.ceil(math.sqrt(n)))
    lo, hi = X.min(0), X.max(0)
    span = np.where(hi - lo > 0, hi - lo, 1.0)
    U = (X - lo) / span
    idx = np.minimum((U * g).astype(int), g - 1)
    counts = np.zeros((g, g))
    np.add.at(counts, (idx[:, 0], idx[:, 1]), 1)
    cbar = n / g ** 2
    num = np.sqrt(np.sum((counts - cbar) ** 2))
    den = np.sqrt((n - cbar) ** 2 + (g ** 2 - 1) * cbar ** 2)
    return float(num / den)


def edge_length_deviation(X: np.ndarray, E: np.ndarray) -> float:
    """Coefficient of variation of edge lengths (scale invariant)."""
    L = np.linalg.norm(X[E[:, 0]] - X[E[:, 1]], axis=1)
    mu = L.mean()
    return float(L.std() / mu) if mu > 0 else 0.0


def mean_edge_length(X: np.ndarray, E: np.ndarray) -> float:
    """Mean edge length after uniform normalisation to the unit box (edge-length scale cue; added v0.1.3)."""
    X = normalise(X)
    return float(np.linalg.norm(X[E[:, 0]] - X[E[:, 1]], axis=1).mean())


def _orient(p, q, r):
    return (q[..., 0] - p[..., 0]) * (r[..., 1] - p[..., 1]) - (q[..., 1] - p[..., 1]) * (r[..., 0] - p[..., 0])


def crossing_count(X: np.ndarray, E: np.ndarray, pairs=None) -> int:
    if pairs is None:
        pairs = nonadjacent_edge_pairs(E)
    a, b = pairs
    if len(a) == 0:
        return 0
    P1, P2 = X[E[a, 0]], X[E[a, 1]]
    Q1, Q2 = X[E[b, 0]], X[E[b, 1]]
    o1, o2 = _orient(P1, P2, Q1), _orient(P1, P2, Q2)
    o3, o4 = _orient(Q1, Q2, P1), _orient(Q1, Q2, P2)
    cross = (o1 * o2 < 0) & (o3 * o4 < 0)  # proper crossings only; collinear touches ignored
    return int(cross.sum())


def crossing_ratio(X: np.ndarray, E: np.ndarray, pairs=None) -> float:
    if pairs is None:
        pairs = nonadjacent_edge_pairs(E)
    cmax = len(pairs[0])
    return crossing_count(X, E, pairs) / cmax if cmax > 0 else 0.0


@dataclass
class GraphCache:
    """Pre-computed arrays for one graph, reused across many drawings."""
    G: nx.Graph
    E: np.ndarray = field(init=False)
    D: np.ndarray = field(init=False)
    iu: np.ndarray = field(init=False)
    ju: np.ndarray = field(init=False)
    pairs: tuple = field(init=False)

    def __post_init__(self):
        self.E, self.D, self.iu, self.ju = graph_arrays(self.G)
        self.pairs = nonadjacent_edge_pairs(self.E)

    @property
    def n(self):
        return self.G.number_of_nodes()


def realism_metrics(X: np.ndarray) -> dict:
    """Shape diagnostics used to reject degenerate drawings (not optimisation targets).
    aspect   = short side / long side of the bounding box (1 = square)
    nn_max   = largest nearest-neighbour distance * sqrt(n)   (large = isolated outlier node)
    nn_min   = smallest pairwise distance * sqrt(n)            (small = overlapping nodes)
    dense    = fraction of nodes inside the densest cell of a 4x4 grid (large = clump)"""
    X = normalise(X)
    n = len(X)
    span = X.max(0) - X.min(0)
    aspect = float(span.min() / span.max()) if span.max() > 0 else 0.0
    Dm = np.linalg.norm(X[:, None] - X[None], axis=-1) + np.eye(n) * 10
    nn = Dm.min(1)
    lo, hi = X.min(0), X.max(0)
    idx = np.minimum(((X - lo) / np.where(hi - lo > 0, hi - lo, 1) * 4).astype(int), 3)
    counts = np.zeros((4, 4)); np.add.at(counts, (idx[:, 0], idx[:, 1]), 1)
    return {"aspect": aspect, "nn_max": float(nn.max() * math.sqrt(n)),
            "nn_min": float(nn.min() * math.sqrt(n)), "dense": float(counts.max() / n)}


REALISM_LIMITS = {"aspect_min": 0.70, "nn_max_max": 2.6, "nn_min_min": 0.20, "dense_max": 0.35}


def realism_ok(r: dict, lim: dict = REALISM_LIMITS) -> bool:
    return (r["aspect"] >= lim["aspect_min"] and r["nn_max"] <= lim["nn_max_max"]
            and r["nn_min"] >= lim["nn_min_min"] and r["dense"] <= lim["dense_max"])


def all_metrics(X: np.ndarray, gc: GraphCache, with_realism: bool = False) -> dict:
    X = normalise(X)
    out = {
        "ksm": ksm(X, gc.D, gc.iu, gc.ju),
        "nu": node_uniformity(X),
        "eld": edge_length_deviation(X, gc.E),
        "xr": crossing_ratio(X, gc.E, gc.pairs),
        "mel": mean_edge_length(X, gc.E),
    }
    if with_realism:
        rm = realism_metrics(X)
        out.update(rm)
        out["realistic"] = realism_ok(rm)
    return out


# ----------------------------------------------------------------------------
# 3. Differentiable surrogates (torch)
# ----------------------------------------------------------------------------
def _torch():
    import torch
    return torch


def t_normalise(X):
    """Differentiable uniform-scale normalisation to the unit box."""
    X = X - X.min(0).values
    return X / (X.max() + 1e-9)


def sur_stress(X, d, iu, ju):
    """Linear-scale stress: sqrt(sum (e - a d)^2 / sum e^2), a = <e,d>/<d,d>. Upper bound of KSM."""
    torch = _torch()
    e = torch.linalg.norm(X[iu] - X[ju], dim=1)
    a = (e * d).sum() / (d * d).sum()
    return torch.sqrt(((e - a * d) ** 2).sum() / (e * e).sum() + 1e-12)


def sur_nu(X, n):
    """Soft-histogram node uniformity on the per-axis bounding box grid."""
    torch = _torch()
    g = int(math.ceil(math.sqrt(n)))
    lo = X.min(0).values
    span = (X.max(0).values - lo).clamp_min(1e-9)
    U = (X - lo) / span
    centres = (torch.arange(g, dtype=X.dtype) + 0.5) / g
    cx, cy = torch.meshgrid(centres, centres, indexing="ij")
    C = torch.stack([cx.reshape(-1), cy.reshape(-1)], 1)          # (g^2, 2)
    sigma = 0.5 / g
    d2 = ((U[:, None, :] - C[None, :, :]) ** 2).sum(-1)           # (n, g^2)
    W = torch.softmax(-d2 / (2 * sigma ** 2), dim=1)
    counts = W.sum(0)
    cbar = n / g ** 2
    num = torch.sqrt(((counts - cbar) ** 2).sum() + 1e-12)
    den = math.sqrt((n - cbar) ** 2 + (g ** 2 - 1) * cbar ** 2)
    return num / den


def sur_eld(X, E):
    torch = _torch()
    L = torch.linalg.norm(X[E[:, 0]] - X[E[:, 1]], dim=1)
    return L.std(unbiased=False) / (L.mean() + 1e-12)


def sur_mel(X, E):
    torch = _torch()
    return torch.linalg.norm(X[E[:, 0]] - X[E[:, 1]], dim=1).mean()


def sur_xr(X, E, pa, pb, k=50.0):
    """Smooth crossing ratio. Orientation tests are normalised (sine of angle) so k is scale free."""
    torch = _torch()
    if len(pa) == 0:
        return X.sum() * 0.0
    P1, P2 = X[E[pa, 0]], X[E[pa, 1]]
    Q1, Q2 = X[E[pb, 0]], X[E[pb, 1]]

    def o(p, q, r):
        v1, v2 = q - p, r - p
        cr = v1[:, 0] * v2[:, 1] - v1[:, 1] * v2[:, 0]
        return cr / (torch.linalg.norm(v1, dim=1) * torch.linalg.norm(v2, dim=1) + 1e-9)

    s1 = -o(P1, P2, Q1) * o(P1, P2, Q2)
    s2 = -o(Q1, Q2, P1) * o(Q1, Q2, P2)
    soft = torch.sigmoid(k * s1) * torch.sigmoid(k * s2)
    return soft.sum() / len(pa)


def sur_realism(X, r_max, lam_aspect=1.0):
    """Penalise outliers (nearest-neighbour distance > r_max) and elongated drawings (aspect ratio far from 1).
    Without this, the optimiser reaches high stress or a target NU through degenerate shapes:
    a few far-away nodes stretching the bounding box, or a thin strip."""
    torch = _torch()
    n = X.shape[0]
    D = torch.cdist(X, X) + torch.eye(n, dtype=X.dtype) * 10.0
    nn = D.min(1).values
    outlier = (torch.relu(nn - r_max) ** 2).sum() / (r_max ** 2)
    span = X.max(0).values - X.min(0).values
    aspect = (span[0] - span[1]) ** 2 / (span.max() ** 2 + 1e-12)
    return outlier + lam_aspect * aspect * n


def sur_overlap(X, r_min=0.03):
    torch = _torch()
    n = X.shape[0]
    iu, ju = torch.triu_indices(n, n, offset=1)
    dist = torch.linalg.norm(X[iu] - X[ju], dim=1)
    return (torch.relu(r_min - dist) ** 2).sum()


# ----------------------------------------------------------------------------
# 4. PDSG Stage 1 — optimise one drawing toward target metric values
# ----------------------------------------------------------------------------
DEFAULT_TOL = {"ksm": 0.01, "nu": 0.02, "eld": 0.02, "xr": 0.01, "mel": 0.01}


@dataclass
class PDSGResult:
    X: np.ndarray
    metrics: dict
    converged: bool
    iters: int
    restarts: int
    seconds: float
    errors: dict


def initial_layout(G, low_stress: bool, rng: np.random.Generator):
    n = G.number_of_nodes()
    if low_stress:
        pos = nx.kamada_kawai_layout(G)
        X = np.array([pos[i] for i in range(n)])
        X = normalise(X) + rng.normal(0, 0.02, size=(n, 2))
    else:
        X = rng.uniform(0, 1, size=(n, 2))
    th = rng.uniform(0, 2 * math.pi)
    R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]])
    return normalise(X @ R.T)


def pdsg_optimise(gc: GraphCache, targets: dict, tol: dict | None = None, X0: np.ndarray | None = None,
                  max_steps: int = 2000, max_restarts: int = 5, lr: float = 0.05, check_every: int = 50,
                  k_xr: float = 50.0, lam_overlap: float = 1.0, spacing_c: float = 0.45, r_min: float | None = None,
                  outlier_c: float = 2.0, lam_realism: float = 1.0,
                  seed: int = 0, verbose: bool = False) -> PDSGResult:
    """
    targets: dict with any of keys 'ksm','nu','eld','xr' -> float target, or None/absent = free.
    Loss = sum_p ((sur_p - t~_p)/tol_p)^2 + lam_overlap * overlap.
    Spacing: overlap penalty with r_min = spacing_c / sqrt(n) (unit box) stops the optimiser reaching
    high stress by collapsing nodes into a clump (a degenerate, trivially detectable drawing).
    Realism: nearest-neighbour distance capped at outlier_c / sqrt(n); aspect ratio pulled toward 1.
    Outer correction: every `check_every` steps the surrogate target t~_p is shifted by (target - exact),
    so the optimiser converges on the EXACT metric even though it differentiates a surrogate.
    """
    torch = _torch()
    tol = {**DEFAULT_TOL, **(tol or {})}
    active = {k: v for k, v in targets.items() if v is not None}
    seed = int(seed)
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    t0 = time.time()

    E_t = torch.as_tensor(gc.E, dtype=torch.long)
    d_t = torch.as_tensor(gc.D[gc.iu, gc.ju], dtype=torch.float64)
    iu_t = torch.as_tensor(gc.iu, dtype=torch.long)
    ju_t = torch.as_tensor(gc.ju, dtype=torch.long)
    pa_t = torch.as_tensor(gc.pairs[0], dtype=torch.long)
    pb_t = torch.as_tensor(gc.pairs[1], dtype=torch.long)
    n = gc.n
    if r_min is None:
        r_min = spacing_c / math.sqrt(n)
    r_max = outlier_c / math.sqrt(n)

    def surrogates(Xt):
        Xn = t_normalise(Xt)
        out = {}
        if "ksm" in active:
            out["ksm"] = sur_stress(Xn, d_t, iu_t, ju_t)
        if "nu" in active:
            out["nu"] = sur_nu(Xn, n)
        if "eld" in active:
            out["eld"] = sur_eld(Xn, E_t)
        if "xr" in active:
            out["xr"] = sur_xr(Xn, E_t, pa_t, pb_t, k_xr)
        if "mel" in active:
            out["mel"] = sur_mel(Xn, E_t)
        return out, Xn

    if X0 is None:
        X0 = initial_layout(gc.G, low_stress=(active.get("ksm", 0.5) <= 0.55), rng=rng)
    best = None
    total_iters = 0
    X_start = np.array(X0, dtype=np.float64)

    for restart in range(max_restarts + 1):
        Xt = torch.tensor(X_start, dtype=torch.float64, requires_grad=True)
        opt = torch.optim.Adam([Xt], lr=lr, betas=(0.9, 0.99))
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_steps, eta_min=lr * 0.1)
        sur_target = dict(active)
        for step in range(1, max_steps + 1):
            opt.zero_grad()
            sur, Xn = surrogates(Xt)
            loss = sum(((sur[k] - sur_target[k]) / tol[k]) ** 2 for k in sur)
            loss = loss + lam_overlap * sur_overlap(Xn, r_min) / (r_min ** 2) + lam_realism * sur_realism(Xn, r_max)
            loss.backward()
            opt.step()
            sched.step()
            total_iters += 1
            if step % check_every == 0 or step == max_steps:
                Xnp = normalise(Xt.detach().numpy())
                m = all_metrics(Xnp, gc)
                err = {k: m[k] - active[k] for k in active}
                score = sum(abs(err[k]) / tol[k] for k in active)
                if best is None or score < best[0]:
                    best = (score, Xnp.copy(), m, err)
                if all(abs(err[k]) < tol[k] for k in active):
                    return PDSGResult(_random_flip(Xnp, rng), m, True, total_iters, restart, time.time() - t0, err)
                for k in active:  # outer correction on surrogate targets
                    sur_target[k] = sur_target[k] - 0.8 * err[k]
                if verbose and step % 500 == 0:
                    print(restart, step, {k: round(v, 4) for k, v in m.items()})
        X_start = best[1] + rng.normal(0, 0.1, size=best[1].shape)

    _, Xb, m, err = best
    return PDSGResult(_random_flip(Xb, rng), m, False, total_iters, max_restarts, time.time() - t0, err)


def _random_flip(X, rng):
    """Random reflection per axis: preserves every metric (incl. NU), unlike arbitrary rotation."""
    X = X.copy()
    if rng.random() < 0.5:
        X[:, 0] = 1 - X[:, 0]
    if rng.random() < 0.5:
        X[:, 1] = 1 - X[:, 1]
    if rng.random() < 0.5:
        X = X[:, ::-1].copy()
    return X


# ----------------------------------------------------------------------------
# 5. Rendering
# ----------------------------------------------------------------------------
RENDER_DEFAULT = dict(res=768, pad=24, node_r=6.0, edge_w=1.5, node_fill="#333333", edge_col="#000000",
                      bg="#FFFFFF", supersample=4, antialias=True)


def render_png(gc: GraphCache, X: np.ndarray, path: str | None = None, **kw):
    """Draw edges then nodes. Returns a PIL image; saves to `path` if given."""
    from PIL import Image, ImageDraw
    o = {**RENDER_DEFAULT, **kw}
    ss = o["supersample"] if o["antialias"] else 1
    R = o["res"] * ss
    pad = o["pad"] * ss
    Xn = normalise(X)
    P = pad + Xn * (R - 2 * pad)
    P[:, 1] = R - P[:, 1]  # y up
    img = Image.new("RGB", (R, R), o["bg"])
    dr = ImageDraw.Draw(img)
    ew = max(1, int(round(o["edge_w"] * ss)))
    for u, v in gc.E:
        dr.line([tuple(P[u]), tuple(P[v])], fill=o["edge_col"], width=ew)
    r = o["node_r"] * ss
    for x, y in P:
        dr.ellipse([x - r, y - r, x + r, y + r], fill=o["node_fill"])
    if ss > 1:
        img = img.resize((o["res"], o["res"]), Image.LANCZOS)
    if path:
        img.save(path, optimize=True)
    return img


# ----------------------------------------------------------------------------
# 6. Parsing model output
# ----------------------------------------------------------------------------
_ANS_RE = re.compile(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', re.I)


def parse_answer(text: str) -> dict:
    """Return {'answer': '1'|'2'|'same'|None, 'confidence': int|None, 'reason': str|None, 'parse_ok': bool}."""
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = _ANS_RE.search(text)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
    return out


# ----------------------------------------------------------------------------
# 7. Statistics helpers
# ----------------------------------------------------------------------------
def bootstrap_ci(values, groups=None, n_boot=10000, ci=95, seed=0, stat=np.mean):
    """Bootstrap CI of `stat`. If `groups` is given, resample groups (e.g., graph ids) with replacement."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if groups is None:
        idx = rng.integers(0, len(values), size=(n_boot, len(values)))
        boots = np.array([stat(values[i]) for i in idx])
    else:
        groups = np.asarray(groups)
        uniq = np.unique(groups)
        members = {g: np.where(groups == g)[0] for g in uniq}
        boots = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.choice(uniq, size=len(uniq), replace=True)
            boots[b] = stat(values[np.concatenate([members[g] for g in pick])])
    lo, hi = np.percentile(boots, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(stat(values)), float(lo), float(hi)


Writing /kaggle/working/n06_output/pdcp_utils.py


## 2. Mean-edge-length counterfactuals for the 400-pair subset (CPU, ≈ 10 min; reused when resuming)

In [4]:
import numpy as np, pandas as pd
sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U; importlib.reload(U); print("utils", U.__version__)
pp = sorted(glob.glob(f"{N03_ROOT}/**/pairs.parquet", recursive=True)); assert pp, "attach the N03 output"
N03 = os.path.dirname(pp[0])
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet")
SUB = PR[(PR.set == "main") & PR.subset400]
prev = glob.glob(f"{RESUME_ROOT}/**/mel_cf_pairs.parquet", recursive=True) if RESUME_ROOT else []
if prev:
    pdir = os.path.dirname(prev[0])
    MCP = pd.read_parquet(f"{pdir}/mel_cf_pairs.parquet"); MCD = pd.read_parquet(f"{pdir}/mel_cf_drawings.parquet")
    for p in MCD.png: shutil.copy(f"{pdir}/{p}", f"{OUTPUT_DIR}/{p}")
    print("reused", len(MCP), "MEL counterfactual pairs from RESUME_ROOT")
else:
    from joblib import Parallel, delayed
    Di = DR.set_index("drawing_id")
    def mel_job(pair, d, seed):
        import sys; sys.path.insert(0, OUTPUT_DIR)
        import numpy as np, torch, pdcp_utils as U; torch.set_num_threads(1)
        a = Di.loc[pair["drawing_A"]]
        gc = U.GraphCache(U.make_graph(a.family, int(a.n), int(a.graph_seed)))
        X0 = np.array(a.coords).reshape(-1, 2); m0 = U.all_metrics(X0, gc)
        tg = {k: m0[k] for k in ["ksm", "nu", "eld", "xr"]}; tg["mel"] = m0["mel"] + d
        r = U.pdsg_optimise(gc, tg, X0=X0, tol={"ksm": 0.004}, max_steps=1200, max_restarts=1, seed=seed)
        m = U.all_metrics(r.X, gc, with_realism=True)
        did = f"CFM_{pair['pair_id']}_mel_{'m' if d < 0 else 'p'}{int(round(abs(d)*100)):02d}"
        ok = bool(r.converged and m["realistic"])
        if ok: U.render_png(gc, r.X, f"{OUTPUT_DIR}/images/768/{did}.png", res=768)
        return {"drawing_id": did, "ok": ok, "base_pair": pair["pair_id"], "delta": d, "family": a.family, "n": int(a.n),
                "graph": int(a.graph), "graph_seed": int(a.graph_seed), "mel_base": m0["mel"],
                **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]},
                "coords": r.X.ravel().round(6).tolist(), "png": f"images/768/{did}.png"}
    rs = np.random.default_rng(SEED)
    jobs = [(p, d, int(rs.integers(1e8))) for p in SUB.to_dict("records") for d in MEL_DELTAS]
    MCD = pd.DataFrame(Parallel(n_jobs=N_CPU, verbose=2)(delayed(mel_job)(*j) for j in jobs))
    print("MEL edit success:", MCD.groupby("delta").ok.mean().round(3).to_dict())
    MCD = MCD[MCD.ok].reset_index(drop=True)
    Bk = DR.set_index("drawing_id")
    rows = []
    for r in MCD.itertuples():
        base = SUB.set_index("pair_id").loc[r.base_pair]; B = Bk.loc[base.drawing_B]
        dk = B.ksm - r.ksm
        rows.append({"pair_id": f"{r.base_pair}_mel{'m' if r.delta < 0 else 'p'}{int(round(abs(r.delta)*100)):02d}", "set": "counterfactual",
                     "drawing_A": r.drawing_id, "drawing_B": base.drawing_B, "family": base.family, "n": int(base.n), "graph": int(base.graph),
                     "dstress_design": base.dstress_design, "cond": base.cond, "d_ksm": dk, "d_nu": B.nu - r.nu, "d_eld": B.eld - r.eld,
                     "d_xr": B.xr - r.xr, "correct": "same" if abs(dk) < 0.005 else ("A" if dk > 0 else "B"), "subset400": True,
                     "cf_proxy": "mel", "cf_delta": r.delta, "base_pair": r.base_pair, "default_order": base.default_order})
    MCP = pd.DataFrame(rows)
    MCD.to_parquet(f"{OUTPUT_DIR}/mel_cf_drawings.parquet", index=False); MCP.to_parquet(f"{OUTPUT_DIR}/mel_cf_pairs.parquet", index=False)
print(len(MCP), "MEL counterfactual pairs |", f"{(time.time()-T0)/60:.1f} min")

utils 0.1.3


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   46.8s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  2.9min
[Parallel(n_jobs=4)]: Done 357 tasks      | elapsed:  5.9min
[Parallel(n_jobs=4)]: Done 640 tasks      | elapsed: 11.9min
[Parallel(n_jobs=4)]: Done 1005 tasks      | elapsed: 19.7min
[Parallel(n_jobs=4)]: Done 1200 out of 1200 | elapsed: 22.3min finished


MEL edit success: {-0.05: 0.932, 0.05: 0.99, 0.1: 0.955}
1151 MEL counterfactual pairs | 22.9 min


## 3. Query list: all counterfactual pairs (N03 NU/ELD/XR + MEL), both orders

In [5]:
png = dict(zip(DR.drawing_id, DR.png))
def path(did):
    return f"{N03}/{png[did]}" if did in png else f"{OUTPUT_DIR}/images/768/{did}.png"
CFP = pd.concat([PR[PR.set == "counterfactual"], MCP], ignore_index=True)
rows = []
for r in CFP.itertuples():
    a, b = path(r.drawing_A), path(r.drawing_B)
    for order in ("AB", "BA"):
        img1, img2 = (a, b) if order == "AB" else (b, a)
        corr = "same" if r.correct == "same" else ("1" if (r.correct == "A") == (order == "AB") else "2")
        rows.append({"pair_id": r.pair_id, "order": order, "img1": img1, "img2": img2, "correct_img": corr,
                     "base_pair": r.base_pair, "cf_proxy": r.cf_proxy, "cf_delta": r.cf_delta, "dstress": r.dstress_design, "cond": r.cond})
Q = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
missing = [p for p in set(Q.img1) | set(Q.img2) if not os.path.exists(p)]
print(len(Q), "queries per model | by proxy:", (Q.cf_proxy.value_counts() // 2).to_dict(), "| missing images:", len(missing))
assert not missing
Q.to_csv(f"{OUTPUT_DIR}/e2_queries.csv", index=False)
if RESUME_ROOT:
    for f in glob.glob(f"{RESUME_ROOT}/**/results/e2/*.jsonl", recursive=True):
        shutil.copy(f, f"{OUTPUT_DIR}/results/e2/{os.path.basename(f)}"); print("resumed", os.path.basename(f), sum(1 for _ in open(f)))

7980 queries per model | by proxy: {'eld': 1151, 'mel': 1151, 'xr': 1119, 'nu': 569} | missing images: 0


## 4. Inference worker v0.3 (unchanged from N05)

In [6]:
%%writefile /kaggle/working/n06_output/pdcp_infer.py
"""
pdcp_infer.py — PDCP VLM inference worker (v0.3.0). One model per process.
v0.3: processor_kwargs API, per-model processor options + default batch, 25-word reasons, early abort on unparseable output.

Usage:
  python pdcp_infer.py --model Q3VL-4B --pairs pairs.csv --out results.jsonl --summary sum.json
                       [--prompt P-Def] [--batch 1] [--batch_test 4] [--max_new 160] [--hf_token TOKEN]

pairs.csv columns: pair_id, order (AB|BA), img1, img2 (absolute PNG paths), correct_img ("1"|"2"|"same")
Resumable: rows already in --out (same pair_id+order+prompt) are skipped.
"""
import argparse, gc, json, os, re, sys, time, traceback

MODELS = {   # batch = default batch size on one T4 (from the N04 pilot); proc = extra processor options
    "Q3VL-2B":     {"ids": ["Qwen/Qwen3-VL-2B-Instruct"], "quant": None, "batch": 8, "proc": {}},
    "Q3VL-4B":     {"ids": ["Qwen/Qwen3-VL-4B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q3VL-8B":     {"ids": ["Qwen/Qwen3-VL-8B-Instruct"], "quant": "nf4", "batch": 2, "proc": {}},
    "Q25VL-7B":    {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "nf4", "batch": 4, "proc": {}},
    "Gemma3-4B":   {"ids": ["google/gemma-3-4b-it"], "quant": None, "batch": 4, "proc": {}},
    "InternVL-4B": {"ids": ["OpenGVLab/InternVL3_5-4B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4,
                    "proc": {"crop_to_patches": False}},   # one 448-px tile per image (pilot: tiling -> 11 s/query, OOM at batch 4)
    "SmolVLM-2B":  {"ids": ["HuggingFaceTB/SmolVLM2-2.2B-Instruct"], "quant": None, "batch": 1, "proc": {}},
}

SYSTEM = ("You are a careful visual analyst. You will be shown two images of node-link diagrams that depict the SAME "
          "network (same nodes and same edges), drawn with different node positions. Answer the question strictly in "
          "the JSON format requested. Do not add any text outside the JSON.")
DEFINITION = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures "
              "how well the straight-line distances between pairs of nodes match the graph-theoretic distances between them "
              "(the number of edges on the shortest path connecting them). A drawing has LOW stress when pairs of nodes that "
              "are few edges apart are drawn close together and pairs that are many edges apart are drawn far apart, "
              "consistently across the whole drawing. A drawing has HIGH stress when these distances disagree.")
JSON_SPEC = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>, '
             '"reason": "<at most 25 words>"}')
PROMPTS = {
    "P-Def": DEFINITION + "\n\nImage 1 and Image 2 show the same network. Which image has LOWER stress?\n\n" + JSON_SPEC,
    "P-Expert": ("You are an expert in graph drawing and network visualization. Image 1 and Image 2 show the same network. "
                 "Which image has lower stress?\n\n" + JSON_SPEC),
}


def parse_answer(text):
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = re.search(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', text, re.I)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
        r = re.search(r'"reason"\s*:\s*"([^"]*)', text)
        out["reason"] = r.group(1) if r else None
    return out


def build_messages(img1, img2, prompt_id, with_images=True):
    def im(x):
        return {"type": "image", "image": x} if with_images else {"type": "image"}
    return [{"role": "user", "content": [
        {"type": "text", "text": SYSTEM + "\n\nImage 1:"}, im(img1),
        {"type": "text", "text": "Image 2:"}, im(img2),
        {"type": "text", "text": PROMPTS[prompt_id]}]}]


class Runner:
    def __init__(self, key, hf_token=None, log=print):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        self.torch, self.key, self.log = torch, key, log
        spec = MODELS[key]
        self.route, self.model, self.proc, self.model_id, self.dtype_name = None, None, None, None, None
        errors = []
        for mid in spec["ids"]:
            for dt in (["float16", "bfloat16"] if spec["quant"] is None else ["float16"]):
                try:
                    t0 = time.time()
                    kw = {"device_map": {"": 0}, "token": hf_token, "low_cpu_mem_usage": True}
                    if spec["quant"] == "nf4":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                        bnb_4bit_use_double_quant=True)
                    else:
                        kw["dtype"] = getattr(torch, dt)
                    if self.model is None or self.model_id != mid:
                        self._free()
                        self.model = AutoModelForImageTextToText.from_pretrained(mid, **kw).eval()
                        self.proc = AutoProcessor.from_pretrained(mid, token=hf_token)
                    self.model_id, self.dtype_name, self.load_s = mid, (spec["quant"] or dt), time.time() - t0
                    return
                except Exception as e:
                    errors.append(f"{mid} {dt}: {type(e).__name__}: {str(e)[:300]}")
                    self.log("load failed: " + errors[-1])
                    self._free()
        raise RuntimeError("all loads failed | " + " || ".join(errors))

    def reload_dtype(self, dt):
        """Reload the same model in another dtype (used when fp16 produces garbage)."""
        from transformers import AutoModelForImageTextToText
        self._free(keep_proc=True)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, device_map={"": 0},
                                                                 dtype=getattr(self.torch, dt), low_cpu_mem_usage=True).eval()
        self.dtype_name = dt

    def _free(self, keep_proc=False):
        if self.model is not None:
            del self.model
            self.model = None
        if not keep_proc:
            self.proc = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass

    def _inputs(self, batch, prompt_id):
        """batch = list of (img1, img2). Tries route A (chat template loads images) then B (processor call)."""
        torch = self.torch
        routes = [self.route] if self.route else ["A", "B_nested", "B_flat"]
        last = None
        for route in routes:
            try:
                if route == "A":
                    msgs = [build_messages(a, b, prompt_id) for a, b in batch]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    pk = dict(MODELS[self.key].get("proc", {}))
                    if len(batch) > 1:
                        pk["padding"] = True
                    inp = self.proc.apply_chat_template(msgs if len(batch) > 1 else msgs[0], add_generation_prompt=True,
                                                        tokenize=True, return_dict=True, return_tensors="pt",
                                                        processor_kwargs=pk)
                else:
                    texts = [self.proc.apply_chat_template(build_messages(a, b, prompt_id, with_images=False),
                                                           add_generation_prompt=True, tokenize=False) for a, b in batch]
                    imgs = [[a, b] for a, b in batch] if route == "B_nested" else [x for ab in batch for x in ab]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc(text=texts, images=imgs, return_tensors="pt", padding=True, **MODELS[self.key].get("proc", {}))
                self.route = route
                dev = self.model.device
                mdt = torch.float16 if self.dtype_name in ("nf4", "float16") else getattr(torch, self.dtype_name)
                return {k: (v.to(dev, dtype=mdt) if torch.is_floating_point(v) else v.to(dev)) if hasattr(v, "to") else v
                        for k, v in inp.items()}
            except Exception as e:
                last = e
                if self.route:
                    raise
        raise RuntimeError(f"no input route worked: {type(last).__name__}: {last}")

    def generate(self, batch, prompt_id, max_new=160):
        torch = self.torch
        inp = self._inputs(batch, prompt_id)
        n_in = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = self.model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None)
        texts = self.proc.batch_decode(out[:, n_in:], skip_special_tokens=True)
        return texts, int(n_in)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--summary", required=True)
    ap.add_argument("--prompt", default="P-Def"); ap.add_argument("--batch", type=int, default=0, help="0 = model default")
    ap.add_argument("--min_parse", type=float, default=0.5, help="abort if parse rate after 40 queries is below this")
    ap.add_argument("--deadline", type=float, default=0, help="unix time after which the worker stops cleanly")
    ap.add_argument("--batch_test", type=int, default=0); ap.add_argument("--max_new", type=int, default=160)
    ap.add_argument("--hf_token", default=None); ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()
    import pandas as pd, torch
    from PIL import Image
    def log(s): print(f"[{a.model} {time.strftime('%H:%M:%S')}] {s}", flush=True)
    summ = {"model": a.model, "prompt": a.prompt, "ok": False, "error": None}
    t_start = time.time()
    try:
        P = pd.read_csv(a.pairs)
        if a.limit: P = P.head(a.limit)
        done = set()
        if os.path.exists(a.out):
            for line in open(a.out):
                try:
                    r = json.loads(line); done.add((r["pair_id"], r["order"], r["prompt_id"]))
                except Exception:
                    pass
        todo = [r for r in P.to_dict("records") if (r["pair_id"], r["order"], a.prompt) not in done]
        log(f"{len(P)} pairs, {len(todo)} to do")
        torch.cuda.reset_peak_memory_stats()
        R = Runner(a.model, a.hf_token, log)
        log(f"loaded {R.model_id} as {R.dtype_name} in {R.load_s:.0f}s")
        summ.update({"model_id": R.model_id, "dtype": R.dtype_name, "load_s": round(R.load_s, 1)})
        load = lambda p: Image.open(p).convert("RGB")

        # warm-up + dtype sanity: if fp16 gives zero parseable answers on 3 items, retry in bf16
        if todo:
            w = todo[:3]
            texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in w[:1]], a.prompt, a.max_new)
            ok = parse_answer(texts[0])["parse_ok"]
            if not ok:
                t2 = [R.generate([(load(r["img1"]), load(r["img2"]))], a.prompt, a.max_new)[0][0] for r in w[1:]]
                if not any(parse_answer(t)["parse_ok"] for t in t2) and R.dtype_name == "float16":
                    log("fp16 output unparseable on warm-up -> retrying in bfloat16. sample: " + repr(texts[0][:200]))
                    try:
                        R.reload_dtype("bfloat16"); summ["dtype"] = "bfloat16"
                    except Exception as e:
                        log(f"bf16 reload failed ({type(e).__name__}); reloading fp16 and continuing")
                        R.reload_dtype("float16"); summ["dtype_note"] = "fp16 unparseable, bf16 unavailable"
        summ["route"] = R.route

        f = open(a.out, "a")
        times, n_ok, n_corr, n = [], 0, 0, 0
        bs = a.batch if a.batch > 0 else MODELS[a.model].get("batch", 1)
        summ["batch"] = bs
        for i in range(0, len(todo), bs):
            chunk = todo[i:i + bs]
            if a.deadline and time.time() > a.deadline:
                log("deadline reached - stopping cleanly (resumable)"); summ["stopped_at_deadline"] = True; break
            t0 = time.time()
            try:
                texts, n_in = R.generate([(load(r["img1"]), load(r["img2"])) for r in chunk], a.prompt, a.max_new)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs > 1:
                    log("OOM at batch %d -> falling back to batch 1" % bs); bs = 1
                    texts, n_in = [], 0
                    for r in chunk:
                        t, n_in = R.generate([(load(r["img1"]), load(r["img2"]))], a.prompt, a.max_new); texts += t
                else:
                    raise
            dt = (time.time() - t0) / len(chunk)
            for r, t in zip(chunk, texts):
                p = parse_answer(t)
                corr = p["answer"] == str(r["correct_img"]) if p["parse_ok"] else None
                rec = {"pair_id": r["pair_id"], "order": r["order"], "model": a.model, "model_id": R.model_id,
                       "dtype": R.dtype_name, "prompt_id": a.prompt, "raw_text": t, "parsed": p, "parse_ok": p["parse_ok"],
                       "correct_img": str(r["correct_img"]), "correct": corr, "latency_s": round(dt, 3), "input_tokens": n_in,
                       "batch": len(chunk), "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
                f.write(json.dumps(rec) + "\n")
                n += 1; n_ok += p["parse_ok"]; n_corr += bool(corr)
            f.flush(); times.append(dt)
            if n >= 40 and n - len(chunk) < 40 and n_ok / n < a.min_parse:
                log(f"parse rate {n_ok/n:.2f} < {a.min_parse} after {n} queries - aborting this model")
                summ["aborted_low_parse"] = True; break
            if n % 50 < len(chunk):
                log(f"{n}/{len(todo)} | {dt:.2f}s/q | parse {n_ok/n:.2f} | acc {n_corr/max(1,n_ok):.2f}")
        f.close()
        summ.update({"n": n, "sec_per_query": round(float(sum(times) / max(1, len(times))), 3), "parse_rate": round(n_ok / max(1, n), 3),
                     "acc_on_parsed": round(n_corr / max(1, n_ok), 3), "input_tokens": n_in if todo else None})

        # optional batched-throughput test on the first items (not written to results)
        if a.batch_test and a.batch_test > 1 and len(P) >= a.batch_test * 3:
            rows = P.to_dict("records")[: a.batch_test * 3]
            try:
                t0 = time.time()
                for i in range(0, len(rows), a.batch_test):
                    texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in rows[i:i + a.batch_test]], a.prompt, a.max_new)
                summ["batch_test"] = {"batch": a.batch_test, "sec_per_query": round((time.time() - t0) / len(rows), 3),
                                      "parse_rate": round(sum(parse_answer(t)["parse_ok"] for t in texts) / len(texts), 3)}
            except Exception as e:
                summ["batch_test"] = {"batch": a.batch_test, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            log(f"batch test: {summ['batch_test']}")
        summ["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        summ["ok"] = True
    except Exception as e:
        summ["error"] = f"{type(e).__name__}: {str(e)[:1500]}"
        log("FAILED: " + summ["error"]); traceback.print_exc()
    summ["wall_s"] = round(time.time() - t_start, 1)
    json.dump(summ, open(a.summary, "w"), indent=2)
    log(f"done: {json.dumps({k: summ.get(k) for k in ['ok', 'n', 'sec_per_query', 'parse_rate', 'acc_on_parsed', 'vram_peak_gb']})}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n06_output/pdcp_infer.py


## 5. Run (two processes in parallel, one per GPU; stops cleanly before the session limit)

In [7]:
HF_CACHE = "/tmp/hf_cache"
import pdcp_infer as I; importlib.reload(I)
DEADLINE = T0 + SESSION_HOURS * 3600
done_n = {m: (sum(1 for _ in open(f"{OUTPUT_DIR}/results/e2/{m}.jsonl")) if os.path.exists(f"{OUTPUT_DIR}/results/e2/{m}.jsonl") else 0) for m in MODELS}
queue = [m for m in MODELS if done_n[m] < len(Q)]; running = {}
print("to run:", queue)
def launch(model, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_HUB_CACHE=HF_CACHE, TOKENIZERS_PARALLELISM="false",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [sys.executable, f"{OUTPUT_DIR}/pdcp_infer.py", "--model", model, "--pairs", f"{OUTPUT_DIR}/e2_queries.csv",
           "--out", f"{OUTPUT_DIR}/results/e2/{model}.jsonl", "--summary", f"{OUTPUT_DIR}/results/e2/{model}.summary.json",
           "--prompt", PROMPT, "--max_new", str(MAX_NEW_TOKENS), "--deadline", str(DEADLINE)]
    if HF_TOKEN: cmd += ["--hf_token", HF_TOKEN]
    logf = open(f"{OUTPUT_DIR}/logs/{model}.log", "a")
    running[gpu] = (model, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env), time.time(), logf)
    print(f"[{(time.time()-T0)/60:6.1f} min] start {model} on GPU {gpu}")
last = 0
while queue or running:
    for gpu in range(N_GPU):
        if gpu not in running and queue and time.time() < DEADLINE - 1800: launch(queue.pop(0), gpu)
    if not running: break
    time.sleep(30)
    if time.time() - last > 1800:
        last = time.time()
        print("   " + " | ".join(f"{m}: {sum(1 for _ in open(f'{OUTPUT_DIR}/results/e2/{m}.jsonl')) if os.path.exists(f'{OUTPUT_DIR}/results/e2/{m}.jsonl') else 0}/{len(Q)}" for m in MODELS))
    for gpu, (model, p, st, logf) in list(running.items()):
        if p.poll() is not None:
            logf.close(); del running[gpu]
            print(f"[{(time.time()-T0)/60:6.1f} min] end {model}: {open(f'{OUTPUT_DIR}/logs/{model}.log').read().strip().splitlines()[-1][-250:]}")
            for mid in I.MODELS[model]["ids"]: shutil.rmtree(f"{HF_CACHE}/models--{mid.replace('/', '--')}", ignore_errors=True)
print(f"run phase finished after {(time.time()-T0)/3600:.2f} h")

to run: ['Q25VL-7B', 'Q3VL-4B', 'Q3VL-2B', 'InternVL-4B']
[  22.9 min] start Q25VL-7B on GPU 0
[  22.9 min] start Q3VL-4B on GPU 1
   Q25VL-7B: 0/7980 | Q3VL-4B: 0/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 244/7980 | Q3VL-4B: 436/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 520/7980 | Q3VL-4B: 912/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 796/7980 | Q3VL-4B: 1388/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 1068/7980 | Q3VL-4B: 1864/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 1344/7980 | Q3VL-4B: 2340/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 1620/7980 | Q3VL-4B: 2820/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 1896/7980 | Q3VL-4B: 3296/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 2172/7980 | Q3VL-4B: 3776/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 2448/7980 | Q3VL-4B: 4256/7980 | Q3VL-2B: 0/7980 | InternVL-4B: 0/7980
   Q25VL-7B: 2724/7980 | Q3VL-4B: 4736/7

## 6. Counterfactual effects: change in order-averaged P(choose A) relative to the E1 base pair

In [8]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
def load_results(root, sub):
    R = []
    for f in glob.glob(f"{root}/**/results/{sub}/*.jsonl", recursive=True):
        R += [json.loads(l) for l in open(f)]
    R = pd.DataFrame(R)
    if len(R):
        a = R.parsed.map(lambda p: p.get("answer"))
        R["choice"] = np.where(a == "same", "same", np.where(a.isin(["1", "2"]), np.where((a == "1") == (R.order == "AB"), "A", "B"), None))
    return R
E2 = load_results(OUTPUT_DIR, "e2"); E1 = load_results(N05_ROOT, "e1")
status = {"queries_per_model": len(Q), "hours": round((time.time() - T0) / 3600, 2),
          "models": {m: int((E2.model == m).sum()) if len(E2) else 0 for m in MODELS}}
status["all_complete"] = all(v >= len(Q) for v in status["models"].values())
json.dump(status, open(f"{OUTPUT_DIR}/e2_status.json", "w"), indent=2)
print(json.dumps(status))
if len(E2) and len(E1):
    pA = lambda df: df[df.choice.isin(["A", "B"])].assign(a=lambda x: (x.choice == "A").astype(float)).groupby(["model", "pair_id"]).a.mean()
    base = pA(E1[E1.pair_id.isin(set(Q.base_pair))]).rename("pA_base").reset_index().rename(columns={"pair_id": "base_pair"})
    cf = pA(E2).rename("pA_cf").reset_index().merge(Q.drop_duplicates("pair_id")[["pair_id", "base_pair", "cf_proxy", "cf_delta", "cond", "dstress"]], on="pair_id")
    X = cf.merge(base, on=["model", "base_pair"])
    X["shift"] = X.pA_cf - X.pA_base
    # per-order flips (answer changed vs the base answer in the same order)
    b1 = E1[["model", "pair_id", "order", "choice"]].rename(columns={"pair_id": "base_pair", "choice": "c_base"})
    F = E2.merge(Q[["pair_id", "order", "base_pair"]], on=["pair_id", "order"]).merge(b1, on=["model", "base_pair", "order"])
    F["flip"] = (F.choice != F.c_base).astype(float)
    flips = F.merge(Q.drop_duplicates("pair_id")[["pair_id", "cf_proxy", "cf_delta"]], on="pair_id").groupby(["model", "cf_proxy", "cf_delta"]).flip.mean()
    EFF = X.groupby(["model", "cf_proxy", "cf_delta"]).agg(n=("shift", "size"), mean_shift=("shift", "mean"),
                                                          se=("shift", lambda s: s.std() / np.sqrt(len(s)))).join(flips).reset_index()
    EFF["z"] = EFF.mean_shift / EFF.se
    EFF.round(4).to_csv(f"{OUTPUT_DIR}/e2_effects.csv", index=False)
    pd.set_option("display.width", 200); print(EFF.round(3).to_string(index=False))
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.2), sharey=True)
    for a, p in zip(ax, ["nu", "eld", "xr", "mel"]):
        for m, g in EFF[EFF.cf_proxy == p].groupby("model"):
            a.errorbar(g.cf_delta, g.mean_shift, yerr=1.96 * g.se, marker="o", capsize=3, label=m)
        a.axhline(0, c="k", lw=.8); a.set_title(f"edit {p.upper()} on drawing A (stress fixed)"); a.set_xlabel("δ")
    ax[0].set_ylabel("Δ P(choose A)  (order-averaged)"); ax[-1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e2_shift_curves.png", dpi=120); plt.show()
print("ALL COMPLETE" if status["all_complete"] else "NOT COMPLETE -> save this output as a dataset, set RESUME_ROOT to it and run again")

{"queries_per_model": 7980, "hours": 11.31, "models": {"Q25VL-7B": 5972, "Q3VL-4B": 7980, "Q3VL-2B": 3808, "InternVL-4B": 0}, "all_complete": false}
   model cf_proxy  cf_delta   n  mean_shift    se  flip      z
Q25VL-7B      eld      0.05 362       0.018 0.021 0.207  0.841
Q25VL-7B      eld      0.10 363       0.048 0.022 0.225  2.145
Q25VL-7B      eld      0.15 355       0.021 0.020 0.198  1.048
Q25VL-7B      mel     -0.05 351       0.091 0.023 0.231  4.039
Q25VL-7B      mel      0.05 371      -0.080 0.022 0.236 -3.615
Q25VL-7B      mel      0.10 349      -0.150 0.024 0.282 -6.341
Q25VL-7B       nu      0.05 332      -0.032 0.022 0.197 -1.428
Q25VL-7B       nu      0.10 209      -0.127 0.029 0.256 -4.413
Q25VL-7B       xr      0.02 371      -0.003 0.021 0.202 -0.129
Q25VL-7B       xr      0.04 351      -0.006 0.023 0.221 -0.250
Q25VL-7B       xr      0.06 341      -0.012 0.022 0.211 -0.532
 Q3VL-2B      eld      0.05 274       0.046 0.027 0.199  1.719
 Q3VL-2B      eld      0.10 265 

In [9]:
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N06 E2 · {status['hours']} h · complete: {status['all_complete']} · " + " | ".join(f"{m}: {v}/{len(Q)}" for m, v in status["models"].items()), ""]
for root, _, files in os.walk(OUTPUT_DIR):
    if "images" in root: continue
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):45s} {os.path.getsize(p)/1024:9.1f} KB")
lines.append(f"images/768/: {len(os.listdir(f'{OUTPUT_DIR}/images/768'))} PNG")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N06 E2 · 11.31 h · complete: False · Q25VL-7B: 5972/7980 | Q3VL-4B: 7980/7980 | Q3VL-2B: 3808/7980 | InternVL-4B: 0/7980

e2_effects.csv                                      1.6 KB
e2_queries.csv                                   1850.0 KB
e2_status.json                                      0.2 KB
fig_e2_shift_curves.png                            96.3 KB
mel_cf_drawings.parquet                          1421.4 KB
mel_cf_pairs.parquet                               69.5 KB
pdcp_infer.py                                      16.3 KB
pdcp_utils.py                                      22.4 KB
logs/Q25VL-7B.log                                  19.8 KB
logs/Q3VL-2B.log                                    5.8 KB
logs/Q3VL-4B.log                                   14.3 KB
results/e2/Q25VL-7B.jsonl                        3579.1 KB
results/e2/Q25VL-7B.summary.json                    0.4 KB
results/e2/Q3VL-2B.jsonl                         3591.2 KB
results/e2/Q3VL-2B.summary.json                     